<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

## Interpret the Multiplier by Changing the Requirement

**Change the calibration target a little and observe how the best attainable score changes.**

Keep the same clock observations and squared-error objective. Only the required correction at one hour changes. For each target, solve again for both settings and the multiplier.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 1 · Change one requirement

The recorded errors are $-2,-1,-1,0$ min at $0,1,2,3$ h. Choose correction $q$ in min and rate $r$ in min/h. For $i=1,\ldots,4$, the errors are $y_i=d_i+q+rt_i$ and the score is $\phi(q,r)=\sum_{i=1}^{4}y_i^2$.

Let $\beta$ be the required correction at one hour, in min. Using numerical values in these units, calibration becomes $h=q+r-\beta=0$, and the Lagrangian is $L=\phi+\nu h$.

The original target is $\beta=1$, which makes that reading exact. At $\beta=1.05$, its remaining error is deliberately 0.05 min. This is a changed requirement, not a more accurate version of the original requirement.

The solver uses the same two slope equations and replaces only the equality's target. For this strictly convex quadratic with a linear equality, the resulting candidate is the unique minimum.


In [ ]:
import numpy as np

OBSERVED_ERRORS = np.array([-2.0, -1.0, -1.0, 0.0])  # min
OBSERVATION_TIMES = np.array([0.0, 1.0, 2.0, 3.0])  # h
CALIBRATION_TIME = 1.0  # h
CHECK_TOLERANCE = 1e-9


def solve_calibration(required_correction):
    coefficients = np.array([[8.0, 12.0, 1.0],
                             [12.0, 28.0, 1.0],
                             [1.0, CALIBRATION_TIME, 0.0]])
    correction, rate, multiplier = np.linalg.solve(
        coefficients, np.array([8.0, 6.0, required_correction]))
    errors = OBSERVED_ERRORS + correction + rate * OBSERVATION_TIMES
    residual = correction + CALIBRATION_TIME * rate - required_correction
    return {"decision": np.array([correction, rate]), "response": errors,
            "objective": float(errors @ errors), "multiplier": float(multiplier),
            "feasible": bool(abs(residual) <= CHECK_TOLERANCE)}


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 2 · Run the same calculation for nearby targets

At target 1.00 min, the best settings are $(1.5,-0.5)$, the score is 0.50 min², and the multiplier is 2. Increasing the target to 1.05 min lowers the best score to about 0.40833 min². Decreasing it to 0.95 min raises the best score to about 0.60833 min².

Each result uses newly optimized settings. Comparing a fixed decision after changing its requirement would answer a different question and could violate calibration.


In [ ]:
print("Target(min)   q(min)   r(min/h)  Score(min²)    nu  Feasible")
for target in (0.95, 1.00, 1.05):
    result = solve_calibration(target)
    correction, rate = result["decision"]
    print(f"{target:11.2f} {correction:8.4f} {rate:10.4f} "
          f"{result['objective']:12.5f} {result['multiplier']:5.2f} "
          f"{result['feasible']}")


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 3 · Use the multiplier as a small-change estimate

Let $\Delta\beta$ be a small target change and let $\Delta V$ be the resulting change in the best attainable score. With the residual convention $h=q+r-\beta$, the multiplier gives

$$
\Delta V\approx-\nu\,\Delta\beta.
$$

At the original target, $\nu=2$. For a target increase of 0.05 min, the estimate is $-2(0.05)=-0.10$ min². The actual change is about $-0.09167$ min². The estimate captures the direction and approximate size, not an exact change for a finite step.

The minus sign comes from how the target appears in $h$. A positive multiplier therefore means that a small increase in this target lowers the best score. This interpretation assumes the optimum varies smoothly near the target, as it does here.


In [ ]:
base = solve_calibration(1.0)
for change in (-0.05, 0.05):
    result = solve_calibration(1.0 + change)
    estimate = -base["multiplier"] * change
    actual_change = result["objective"] - base["objective"]
    print(f"Target change {change:+.2f} min: settings {np.round(result['decision'], 4)}, "
          f"feasible = {result['feasible']}")
    print(f"  Estimated score change {estimate:+.5f}, actual {actual_change:+.5f} min²")


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

At target 1.3 min, the equality passes through the unrestricted minimum $(1.9,-0.6)$. The score is 0.2 min² and the multiplier is zero. The equality still holds; its target has no first-order effect on the best score at that point.


In [ ]:
result = solve_calibration(1.3)
print("Settings:", result["decision"])
print("Score:", round(result["objective"], 5), "min²")
print("Multiplier is zero within tolerance:", abs(result["multiplier"]) <= CHECK_TOLERANCE)
print("Calibration holds:", result["feasible"])
